In [1]:
%reset -f

In [2]:
%reset -f

In [3]:
%reset -f

In [4]:
# stop 

In [5]:
!nvidia-smi

Sun Oct 18 11:56:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   43C    P8              3W /   95W |     999MiB /   8188MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 1) Setup

In [6]:
import os 
os.chdir('D:/College/AI/LLM Learning/Module 0/Module-0-Transformers-from-Scratch')

os.getcwd()

'D:\\College\\AI\\LLM Learning\\Module 0\\Module-0-Transformers-from-Scratch'

In [7]:
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)

# 2) Packages

In [8]:
# =====================================================================
# 1. Standard Library & Utilities
# =====================================================================
import os
import sys
import math
import time
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from collections import Counter
from datasets import load_dataset, Dataset

In [9]:
# =====================================================================
# 2. General Data Manipulation, Science & Visualization
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [10]:
# =====================================================================
# 3. Deep Learning & PyTorch Core
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR

In [11]:
# =====================================================================
# 4. Tokenization & NLP Metrics
# =====================================================================
# Custom subword tokenization from scratch (no pretrained weights)
# If using Hugging Face's tokenizers library:
try:
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
except ImportError:
    pass

# Metric computation (ROUGE scores for abstractive summarization)
try:
    from rouge_score import rouge_scorer
except ImportError:
    pass

In [12]:
# =====================================================================
# 5. Device Verification & Seeding Utility
# =====================================================================
def set_seed(seed) -> None:
    # random.seed(seed)
    # np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(seed=seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

# Hardware / Device Status
print("-" * 50)
try:
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU only"
    print(f"CUDA Available: {cuda_available}")
    print(f"Active Device:  {device_name}")
except ImportError:
    print("PyTorch not installed.")
print("-" * 50)

Using device: cuda
Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.0 GB
--------------------------------------------------
CUDA Available: True
Active Device:  NVIDIA GeForce RTX 4060 Laptop GPU
--------------------------------------------------


# 3) Load Dataset

In [13]:
# =====================================================================
# Load Real SAMSum Dataset
# =====================================================================
RAW_DATA_DIR = os.path.join(os.getcwd(), "data", "raw")
DATASET_PATH = os.path.join(RAW_DATA_DIR, "samsum_train.json")
HF_DATASET_NAME = "knkarthick/samsum"

os.makedirs(RAW_DATA_DIR, exist_ok=True)

try:
    # if not os.path.exists(DATASET_PATH):
    #     raise FileNotFoundError(f"Local file not found at {DATASET_PATH}")
    
    # Read directly into Pandas DataFrames (instant & 100% offline)
    train_df = pd.read_json(os.path.join(RAW_DATA_DIR, "samsum_train.json"))
    val_df   = pd.read_json(os.path.join(RAW_DATA_DIR, "samsum_val.json"))
    test_df  = pd.read_json(os.path.join(RAW_DATA_DIR, "samsum_test.json"))
    print("Loaded SAMSum from local folder")

except Exception:
    print(f"Downloading from Hugging Face: {HF_DATASET_NAME} ...")
    raw_datasets = load_dataset(HF_DATASET_NAME)
    
    # Rename 'dialogue' -> 'article' to preserve compatibility with your code
    raw_datasets = raw_datasets.rename_column("dialogue", "article")
    
    # Save local copies to data/raw/
    for split_name, file_name in [("train", "samsum_train.json"), 
                                  ("validation", "samsum_val.json"), 
                                  ("test", "samsum_test.json")]:
        split_records = [dict(item) for item in raw_datasets[split_name]]
        with open(os.path.join(RAW_DATA_DIR, file_name), "w", encoding="utf-8") as f:
            json.dump(split_records, f, indent=2, ensure_ascii=False)
    print("Downloaded and saved SAMSum splits locally!")

    # Standardize: 'dataset' variable pointing to the training split
    train_df = pd.DataFrame(raw_datasets["train"])
    val_df   = pd.DataFrame(raw_datasets["validation"])
    test_df  = pd.DataFrame(raw_datasets["test"])

print("\n" + "=" * 50)
print(f"Total Train Examples: {len(train_df):,}")
print(f"Columns: {train_df.columns}")
print("=" * 50)

# Verify sample
sample = train_df.iloc[0]
print(f"[article / dialogue]:\n{sample['article']}\n")
print(f"[summary]:\n{sample['summary']}")


Loaded SAMSum from local folder

Total Train Examples: 14,731
Columns: Index(['id', 'article', 'summary'], dtype='str')
[article / dialogue]:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

[summary]:
Amanda baked cookies and will bring Jerry some tomorrow.


In [14]:
train_df

,id,article,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14726,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14727,13828570,Theresa: <file_photo>\nTheresa: <file_photo>\n...,Theresa is at work. She gets free food and fre...
14728,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14729,13828395,Jennifer: Dear Celia! How are you doing?\nJenn...,Celia couldn't make it to the afternoon with t...


In [15]:
test_df

,id,article,summary
0,13862856,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...
1,13729565,Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...,Eric and Rob are going to watch a stand-up on ...
2,13680171,"Lenny: Babe, can you help me with something?\n...",Lenny can't decide which trousers to buy. Bob ...
3,13729438,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...
4,13828600,"Ollie: Hi , are you in Warsaw\nJane: yes, just...",Jane is in Warsaw. Ollie and Jane has a party....
...,...,...,...
814,13611902-1,Alex: Were you able to attend Friday night's b...,Benjamin didn't come to see a basketball game ...
815,13820989,Jamilla: remember that the audition starts at ...,The audition starts at 7.30 P.M. in Antena 3.
816,13717193,"Marta: <file_gif>\nMarta: Sorry girls, I click...","Marta sent a file accidentally,"
817,13829115,Cora: Have you heard how much fuss British med...,There was a meet-and-greet with James Charles ...


In [16]:
val_df

,id,article,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\nJackie: but she d...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\nMarla: look what I found ...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...
...,...,...,...
813,13829423,Carla: I've got it...\nDiego: what?\nCarla: my...,Carla's date for graduation is on June 4th. Di...
814,13727710,"Gita: Hello, this is Beti's Mum Gita, I wanted...",Bev is going on the school trip with her son. ...
815,13829261,"Julia: Greg just texted me\nRobert: ugh, delet...",Greg cheated on Julia. He apologises to her. R...
816,13680226,"Marry: I broke my nail ;(\nTina: oh, no!\nMarr...",Marry broke her nail and has a party tomorrow....


In [17]:
train_df.iloc[0]

id                                                  13818513
article    Amanda: I baked  cookies. Do you want some?\nJ...
summary    Amanda baked cookies and will bring Jerry some...
Name: 0, dtype: str

In [18]:
train_df.iloc[0]['article']

"Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)"

In [19]:
train_df.iloc[0]['summary']

'Amanda baked cookies and will bring Jerry some tomorrow.'

In [20]:
train_df.iloc[4]['article']

"Sam: hey  overheard rick say something\nSam: i don't know what to do :-/\nNaomi: what did he say??\nSam: he was talking on the phone with someone\nSam: i don't know who\nSam: and he was telling them that he wasn't very happy here\nNaomi: damn!!!\nSam: he was saying he doesn't like being my roommate\nNaomi: wow, how do you feel about it?\nSam: i thought i was a good rommate\nSam: and that we have a nice place\nNaomi: that's true man!!!\nNaomi: i used to love living with you before i moved in with me boyfriend\nNaomi: i don't know why he's saying that\nSam: what should i do???\nNaomi: honestly if it's bothering you that much you should talk to him\nNaomi: see what's going on\nSam: i don't want to get in any kind of confrontation though\nSam: maybe i'll just let it go\nSam: and see how it goes in the future\nNaomi: it's your choice sam\nNaomi: if i were you i would just talk to him and clear the air"

In [21]:
train_df.iloc[4]['summary']

'Sam is confused, because he overheard Rick complaining about him as a roommate. Naomi thinks Sam should talk to Rick. Sam is not sure what to do.'

# 4) PreProcess Dataset

In [22]:
import re
import pandas as pd
from datasets import DatasetDict

In [23]:
# 1. Convert to pandas DataFrame for inspection & filtering
train_df = pd.DataFrame(train_df)
val_df = pd.DataFrame(val_df)
test_df = pd.DataFrame(test_df)

train_df

,id,article,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14726,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14727,13828570,Theresa: <file_photo>\nTheresa: <file_photo>\n...,Theresa is at work. She gets free food and fre...
14728,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14729,13828395,Jennifer: Dear Celia! How are you doing?\nJenn...,Celia couldn't make it to the afternoon with t...


In [24]:
# 2. Text normalization helper
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Standardize whitespace while preserving sentence punctuation
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()
    
train_df["article"] = train_df["article"].apply(clean_text)
train_df["summary"] = train_df["summary"].apply(clean_text)

val_df["article"] = val_df["article"].apply(clean_text)
val_df["summary"] = val_df["summary"].apply(clean_text)

test_df["article"] = test_df["article"].apply(clean_text)
test_df["summary"] = test_df["summary"].apply(clean_text)

train_df

,id,article,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJe...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i ...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14726,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14727,13828570,Theresa: <file_photo>\nTheresa: <file_photo>\n...,Theresa is at work. She gets free food and fre...
14728,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14729,13828395,Jennifer: Dear Celia! How are you doing?\nJenn...,Celia couldn't make it to the afternoon with t...


In [25]:
# 3. Filtering missing / degenerate entries

train_df = train_df.dropna(subset=["article", "summary"])
train_df = train_df[(train_df["article"].str.split().str.len() >= 10) & 
                    (train_df["summary"].str.split().str.len() >= 3)]

test_df = test_df.dropna(subset=["article", "summary"])
test_df = test_df[(test_df["article"].str.split().str.len() >= 10) & 
                (test_df["summary"].str.split().str.len() >= 3)]

val_df = val_df.dropna(subset=["article", "summary"])
val_df = val_df[(val_df["article"].str.split().str.len() >= 10) & 
                (val_df["summary"].str.split().str.len() >= 3)]

train_df

,id,article,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJe...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i ...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14726,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14727,13828570,Theresa: <file_photo>\nTheresa: <file_photo>\n...,Theresa is at work. She gets free food and fre...
14728,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14729,13828395,Jennifer: Dear Celia! How are you doing?\nJenn...,Celia couldn't make it to the afternoon with t...


In [26]:
# 4. Deduplicate

train_df = train_df.drop_duplicates(subset=["article", "summary"]).reset_index(drop=True)
test_df = test_df.drop_duplicates(subset=["article", "summary"]).reset_index(drop=True)
val_df = val_df.drop_duplicates(subset=["article", "summary"]).reset_index(drop=True)

train_df

,id,article,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJe...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i ...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14680,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14681,13828570,Theresa: <file_photo>\nTheresa: <file_photo>\n...,Theresa is at work. She gets free food and fre...
14682,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14683,13828395,Jennifer: Dear Celia! How are you doing?\nJenn...,Celia couldn't make it to the afternoon with t...


NLP Preprocessing: Classical NLP vs. Our Transformers (Steps Skipped)

| Preprocessing Step | Classical NLP (TF-IDF / ML) | For Our Transformer | Status | Rationale for Sequence-to-Sequence Generation |
| :--- | :---: | :---: | :---: | :--- |
| **Whitespace Stripping** | ✅ Yes | ✅ **YES** | **Done** | Removes erratic spacing, tabs, and duplicate newlines. |
| **HTML / URLs / Emails** | ✅ Yes | ✅ **Optional / Handled** | **Done** | Standardizes noise and media tags without breaking text flow. |
| **Remove Punctuation** | ✅ Yes | ❌ **SKIP (DO NOT DO)** | **Skipped** | Model requires `.`, `?`, `!`, `,` to learn sentence boundaries and syntax for fluent generation. |
| **Remove Numbers** | ✅ Yes | ❌ **SKIP** | **Skipped** | Numbers carry critical factual details (times, dates, quantities) essential for summaries. |
| **Stop Word Removal** | ✅ Yes | ❌ **SKIP (CRITICAL)** | **Skipped** | Removing words like *"the, is, to, will"* forces the model to generate unnatural, broken text. |
| **Stemming / Lemmatization** | ✅ Yes | ❌ **SKIP (CRITICAL)** | **Skipped** | Chopping words to crude roots (`baked` $\rightarrow$ `bak`) destroys grammatical tense and fluency. |
| **Lowercasing (Aggressive)** | ✅ Yes | ❌ **SKIP (Keep Case)** | **Skipped** | Subword tokenizers (BPE) preserve capitalization to distinguish proper nouns and sentence starts. |
| **POS Tagging / NER** | ✅ Yes | ❌ **SKIP** | **Skipped** | Transformer Multi-Head Self-Attention learns syntax, grammar, and entity roles implicitly. |
| **Bag-of-Words / TF-IDF** | ✅ Yes | ❌ **SKIP** | **Skipped** | Replaced by trainable **Token Embeddings + Positional Encodings**. |
| **Subword Tokenization (BPE)** | ❌ No | ✅ **YES (Step 5)** | **Next Step** | Handles out-of-vocabulary words and morphological prefixes/suffixes smoothly. |


# 5) Tokenization and Numerical Encoding

In [27]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, decoders

In [28]:
# =====================================================================
# 1. Build and Train Custom Subword Tokenizer from Scratch
# =====================================================================

##### Example Sentence -> "Amanda  baked  cookies  for  Jerry!"

# Initialize WordPiece subword model with unknown token fallback
tokenizer = Tokenizer(models.WordPiece(unk_token="<UNK>"))

In [29]:
# =====================================================================
# STEP 1: Normalizer (normalizers.NFKC)
# =====================================================================

# 1. Standardizes text characters, formats, and odd Unicode symbols across all rows.
# 2. Prevents duplicate vocabulary entries caused by curly quotes, ligatures, or encoding differences.
#           So it basically removes very extreme characters like ½, ﬁ, etc 
#           But in our example we don't have any

##### Example State: "Amanda  baked  cookies  for  Jerry!" -> "Amanda  baked  cookies  for  Jerry!"

tokenizer.normalizer = normalizers.NFKC()

In [30]:
# =====================================================================
# STEP 2: Pre-Tokenizer (pre_tokenizers.Whitespace)
# =====================================================================

# 1. Splits continuous sentence strings into initial word and punctuation chunks.
#           Basically sentence.split() at whitespaces or punctuations

# 2. Defines clear boundaries so subword merging only happens within individual words.
#             A subword algorithm bridges the gap between Word-level and Character-level tokenization:
#             +-------------------+-------------------------------------+--------------------------------------------------------+
#             | Level             | Example                             | Pros / Cons                                            |
#             +-------------------+-------------------------------------+--------------------------------------------------------+
#             | Word-level        | "unbelievable" -> ["unbelievable"]  | Retains meaning, but unknown words become <UNK>.       |
#             | Character-level   | "cat" -> ['c', 'a', 't']            | Tiny vocab, but sequences become 5x-10x longer.        |
#             | Subword-level     | "unbelievable" -> ["un", "##believ",| Best of both: common words stay whole, rare words split|
#             | (WordPiece/BPE)   |                   "##able"]         | into subwords. Virtually eliminates <UNK> tokens!      |
#             +-------------------+-------------------------------------+--------------------------------------------------------+
#             Why Pre-Tokenizer runs FIRST:
#             1. It puts up "fences" around each word using whitespace/punctuation: ["Amanda", "baked", "cookies"].
#             2. This prevents subword algorithms from accidentally merging characters across different words (e.g. "d" + "coo").

##### Example State: ["Amanda", "baked", "cookies", "for", "Jerry", "!"]

tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [ ]:
# =====================================================================
# STEP 3: Special Tokens (<PAD>, <UNK>, <SOS>, <EOS>)
# =====================================================================

# 1. Reserves fixed IDs (0-3) for structural control signals required by the Transformer.
# 2. <PAD> aligns batch lengths, <SOS> initiates summary decoding, and <EOS> signals decoding completion.

##### Example State: Target Summary -> "<SOS> Amanda baked cookies for Jerry! <EOS>"
#####                IDs            -> [2, 2065, 7874, 4460, 1000, 2502, 4, 3]

SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
VOCAB_SIZE = 8000

In [ ]:
# =====================================================================
# STEP 4: Subword Model & Trainer (WordPieceTrainer)
# =====================================================================

# 1. Learns the top 8,000 most frequent subwords and assigns an integer ID to each.
# 2. Breaks rare/unseen words into subword pieces (using '##') to eliminate unknown words.
#           Basically convert word to vocab position numbers

##### Example State: Tokens -> ['Amanda', 'baked', 'cookies', 'for', 'Jerry', '!']
#####                IDs    -> [  2065,     7874,    4460,     1000,   2502,   4 ]

trainer = trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE,
                                    special_tokens=SPECIAL_TOKENS, 
                                    min_frequency=2)

NameError: name 'SPECIAL_TOKENS' is not defined

In [ ]:
# =====================================================================
# STEP 5: Decoder (decoders.WordPiece)
# =====================================================================

# 1. Reverses tokenization by stitching subword pieces and spaces back together.
# 2. Used during text generation so the model's predicted token IDs become human-readable summaries.

# =====================================================================
# WHERE DOES TOKENIZER.DECODER FIT IN THE PIPELINE?
# =====================================================================
#
#  1. USER INPUT (Text)
#     "Amanda baked chocolatechipcookies for Jerry!"
#                       │
#                       │       [ Tokenizer ENCODES: Text -> IDs ]
#                       ▼
#     Input Token IDs: [2065, 7874, 3102, 5841, 1000, 2502, 4]
#                       │
#                       │       [ Transformer Model (Forward Pass) ]
#                       ▼
#     Output Token IDs: [2065, 7874, 3102, 5841, 1000, 2502, 4]
#                       │
#                       │       [ Tokenizer DECODER: IDs -> Clean Text ] <── CURRENT STEP IS USED HERE!
#                       │               • Glues subwords: "chocolate" + "##chip" -> "chocolatechip"
#                       │               • Fixes punctuation: "Jerry" + "!" -> "Jerry!"
#                       │
#                       ▼ 
#  2. HUMAN-READABLE OUTPUT (Text)
#     "Amanda baked chocolatechipcookies for Jerry!"
#
# * Note: During model training, only Numbers (IDs) are used to compute loss.
#         The Decoder is only used during Evaluation (ROUGE) and User Inference.
# =====================================================================

##### Example State: [2065, 7874, 4460, 1000, 2502, 4] -> "Amanda baked cookies for Jerry!"

tokenizer.decoder = decoders.WordPiece()

In [ ]:
# =====================================================================
# STEP 6: In-Memory Training Iterator & Vocabulary Learning
# =====================================================================
#
# PART 1: WHAT DOES THIS STEP ACTUALLY DO TO OUR TEXT?
# ---------------------------------------------------------------------
# • Up to Step 5, the tokenizer was just an EMPTY BLUEPRINT. It had rules,
#   but its vocabulary had 0 words. It didn't know English!
# • Step 6 is the "Dictionary Author" (Lexicographer) phase:
#   1. It reads all 14,000+ conversations and summaries from train_df.
#   2. It counts letter combinations (e.g. how often does "th", "ing", or
#      "cookies" appear across all training samples).
#   3. It keeps the top 8,000 most frequent words and subword pieces,
#      assigns each a permanent ID from 0 to 7999, and finishes the dictionary.
# • NOTE: This step does NOT modify train_df. It reads from train_df
#   to build the vocabulary dictionary.
#
# ---------------------------------------------------------------------
# PART 2: LINE-BY-LINE CODE BREAKDOWN
# ---------------------------------------------------------------------
# • def train_text_iterator(batch_size=1000):
#   Instead of dumping all 14,000+ texts into RAM at once, this function
#   feeds the text in bite-sized chunks of 1,000 rows.
#
# • for i in range(0, len(train_df), batch_size):
#   Loops through train_df in increments of 1,000 (0-1000, 1000-2000, etc.).
#
# • articles = train_df["article"].iloc[...].tolist()
#   summaries = train_df["summary"].iloc[...].tolist()
#   Extracts 1,000 rows and converts Pandas columns to Python lists of strings.
#
# • yield articles + summaries
#   Uses 'yield' instead of 'return' to create a Python Generator (Stream).
#   It delivers 2,000 text strings (1000 articles + 1000 summaries), pauses,
#   waits for the tokenizer to read them, and then fetches the next batch.
#   This keeps RAM usage extremely low.
#
# • tokenizer.train_from_iterator(train_text_iterator(), trainer=trainer)
#   Where the actual training happens! The engine streams batches from our
#   generator, runs the WordPiece algorithm, counts frequencies, merges
#   character pairs, and creates the 8,000-token vocabulary.
# =====================================================================

##### Example State: Streams 14,000+ training rows in chunks of 1,000 into the vocabulary builder.

def train_text_iterator(batch_size=1000):
    for i in range(0, len(train_df), batch_size):
        articles = train_df["article"].iloc[i:i + batch_size].tolist()
        summaries = train_df["summary"].iloc[i:i + batch_size].tolist()
        yield articles + summaries
        
print("Training custom subword tokenizer strictly on train_df...")
tokenizer.train_from_iterator(train_text_iterator(), trainer=trainer)
print(f"Tokenizer trained successfully! \nVocabulary size: {tokenizer.get_vocab_size():,}")


In [ ]:
# ====================================================================================================
# DEEP DIVE: HOW DOES WORDPIECE / BPE DISCOVER SUBWORDS WITHOUT EXPENSIVE PERMUTATIONS?
# ====================================================================================================
#
# [QUESTION]:
# If words like "playing", "baking", and "running" contain subwords like "ing", how does the tokenizer
# discover that "ing" is a recurring pattern? Doesn't it have to test every possible combination
# of substrings within every word? Wouldn't that lead to a combinatorial explosion that is
# computationally impossible to run in just 2 seconds?
#
# [ANSWER]:
# Subword algorithms (WordPiece / BPE) DO NOT work top-down using brute-force permutations (O(2^N)).
# Instead, they work BOTTOM-UP using an efficient algorithm called: GREEDY PAIR MERGING (O(N)).
#
# ----------------------------------------------------------------------------------------------------
# STEP-BY-STEP EXAMPLE: Suppose our dataset has words like: "playing", "baking", "running"
# ----------------------------------------------------------------------------------------------------
#
# STEP 1: Start at the Very Bottom (Individual Characters)
# Every word in the corpus is initially split into single letters:
#   • "playing" -> ['p', '##l', '##a', '##y', '##i', '##n', '##g']      
#   • "baking"  -> ['b', '##a', '##k',        '##i', '##n', '##g']    
#   • "running" -> ['r', '##u', '##n', '##n', '##i', '##n', '##g']      
# Initial Vocabulary: Just the basic alphabet ('a', 'b', 'c', ... 'z').
# Note : The ## before any word shows that this group can't be start of word, needs something before it 
#        So "in" can be for "independent" or the standalone word "in"
#        However, "##in" has to after a letter inside another word like "playing" or "mint"
#
# STEP 2: Count ADJACENT Pairs Only (Fast Linear Scan)
# The algorithm does NOT search all combinations. It only tallies adjacent neighbors:
#   • ('p', '##l')   -> seen 1 time
#   • ('##i', '##n') -> seen 3 times
#   • ('##n', '##g') -> seen 3 times
# Counting adjacent pairs is a simple O(N) scan—lightning fast across any text.
#
# STEP 3: Merge the #1 Most Frequent Adjacent Pair
# Across thousands of training words, the adjacent pair ('##n', '##g') occurs most frequently.
# The algorithm merges them into a single new token: '##ng'.
# The words in the dataset now become:
#   • ['p', '##l', '##a', '##y', '##i', '##ng']
#   • ['b', '##a', '##k',        '##i', '##ng']
#   • ['r', '##u', '##n', '##n', '##i', '##ng']
# New token '##ng' is added to the vocabulary!
#
# STEP 4: Repeat! (The Birth of "##ing")
# In the next iteration, adjacent pairs are counted again.
# The pair ('##i', '##ng') is now ubiquitous across all continuous verbs!
# It merges: '##i' + '##ng' -> '##ing'.
# Without any manual grammar rules or permutations, '##ing' naturally emerges as a subword token!
#
# STEP 5: Stopping Condition
# This merge process repeats greedily:
#   Merge 1     -> '##ng'
#   Merge 2     -> '##ing'
#   Merge 3     -> 'th'
#   Merge 4     -> 'the'
#   ...
#   Merge 7,996 -> 'cookies'
# The moment the vocabulary reaches exactly VOCAB_SIZE (8,000 tokens), it stops!
#
# ----------------------------------------------------------------------------------------------------
# WHY DOES THIS TAKE ONLY ~2 SECONDS?
# ----------------------------------------------------------------------------------------------------
# 1. Zero Permutations: Only adjacent pairs are counted at each step.
# 2. Hash Tables: Frequencies are tracked using high-speed integer hash counters.
# 3. Rust Backend: The Hugging Face 'tokenizers' library is compiled in optimized, multi-threaded
#    Rust/C++, scanning 15,000+ conversation rows across all CPU cores simultaneously.
# ====================================================================================================


In [ ]:
# =====================================================================
#                               Sample Output
# =====================================================================

PAD_IDX = tokenizer.token_to_id("<PAD>")
UNK_IDX = tokenizer.token_to_id("<UNK>")
SOS_IDX = tokenizer.token_to_id("<SOS>")
EOS_IDX = tokenizer.token_to_id("<EOS>")

print("Special Token IDs:")
print(f"  <PAD> -> {PAD_IDX}")
print(f"  <UNK> -> {UNK_IDX}")
print(f"  <SOS> -> {SOS_IDX}")
print(f"  <EOS> -> {EOS_IDX}")

# Test with a sample from train_df
sample_text = train_df.iloc[0]["summary"]
encoded = tokenizer.encode(sample_text)

print("\n" + "=" * 60)
print(f"Original Text: {sample_text}")
print(f"Tokens:        {encoded.tokens}")
print(f"Token IDs:     {encoded.ids}")
print(f"Decoded Text:  {tokenizer.decode(encoded.ids)}")
print("=" * 60)


In [ ]:
# =====================================================================
# 2. Save Tokenizer Artifact
# =====================================================================
ARTIFACTS_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TOKENIZER_PATH = os.path.join(ARTIFACTS_DIR, "tokenizer.json")
tokenizer.save(TOKENIZER_PATH)
print(f"✅ Tokenizer saved permanently to: {TOKENIZER_PATH}")


# 6) DataLoader Class

# 7) Model

# 8) Training

# 9) Validation

# 10) Outputs